# 02 — Tiền xử lý dữ liệu cho PhoBERT

Notebook này trình bày phần **preprocessing pipeline** của dự án NLP để tiền xử lý dataset đầu vào cả khi train và inference

Mục tiêu:
- Làm sạch review tiếng Việt;
- Chuẩn hóa teencode/emoji/emoticon/Từ tiếng Anh bằng dictionary;
- Tách từ theo chuẩn phù hợp với PhoBERT;
- Tạo cột `phobert_text`;
- Giữ nguyên các cột aspect để train ABSA.

## 1. Khởi tạo môi trường

Notebook có thể chạy từ thư mục `notebooks/` hoặc từ thư mục gốc project.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "src").exists() and (p / "data").exists():
            return p
    if cur.name.lower() == "notebooks":
        return cur.parent
    return cur

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src.preprocessing.pipeline import PreprocessingPipeline
pipeline = PreprocessingPipeline()

PROJECT_ROOT: C:\Users\ADMIN\Downloads\Food_Review_NLP


## 2. Ý tưởng pipeline

Pipeline được thiết kế theo thứ tự:

```text
raw review
   ↓
rule-based cleaning
   ↓
dictionary-based normalization
   ↓
Vietnamese word segmentation
   ↓
PhoBERT-ready text
```

Nguyên tắc:
- Không xóa stopword vì các từ như `không`, `chưa`, `nhưng` rất quan trọng để đánh giá điểm cảm xúc
- Không lowercase toàn bộ vì PhoBERT có thể dùng thông tin chữ hoa
- Không xóa emoji/dấu câu cảm xúc vì chúng có thể biểu diễn sentiment
- Giữ nguyên nhãn aspect `0/1/2/3`

## 3. Demo tiền xử lý trên một số câu review

Cell dưới đây cho thấy sự khác biệt giữa văn bản gốc và văn bản đầu vào cho PhoBERT.

In [2]:
examples = [
    "Đồ ăn không     ngon, giá hợp lý nhưng phục vụ hơi chậm.",
    "Không gian đẹp, nhân viên nhiệt tình, sẽ quay lại ^^",
    "Trà sữa hơi ngọt, giá hơi mắc nhưng view rất xinh!",
    "Món bò bít tết ngon mê mẩn 😍😍😍",
]

rows = []
for text in examples:
    result = pipeline.process(text)
    rows.append({
        "raw": result.get("raw", text),
        "cleaned": result.get("cleaned", ""),
        "normalized": result.get("normalized", ""),
        "phobert_text": result.get("phobert_text", ""),
    })

display(pd.DataFrame(rows))

,raw,cleaned,normalized,phobert_text
0,"Đồ ăn không ngon, giá hợp lý nhưng phục vụ hơi chậm.","Đồ ăn không ngon, giá hợp lý nhưng phục vụ hơi chậm.","Đồ ăn không_ngon , giá hợp lý nhưng phục vụ hơi chậm .","Đồ ăn_không_ngon , giá hợp_lý nhưng phục_vụ hơi chậm ."
1,"Không gian đẹp, nhân viên nhiệt tình, sẽ quay lại ^^","Không gian đẹp, nhân viên nhiệt tình, sẽ quay lại ^^","không_gian đẹp , nhân viên nhiệt tình , sẽ quay lại vui","không_gian đẹp , nhân_viên nhiệt_tình , sẽ quay lại vui"
2,"Trà sữa hơi ngọt, giá hơi mắc nhưng view rất xinh!","Trà sữa hơi ngọt, giá hơi mắc nhưng view rất xinh!","Trà sữa hơi ngọt , giá hơi mắc nhưng cảnh_quan rất xinh !","trà_sữa hơi ngọt , giá hơi mắc nhưng cảnh_quan rất xinh !"
3,Món bò bít tết ngon mê mẩn 😍😍😍,Món bò bít tết ngon mê mẩn 😍😍😍,Món bò bít tết ngon mê mẩn mê_mẩn mê_mẩn mê_mẩn,Món bò_bít_tết ngon_mê mẩn mê_mẩn mê_mẩn mê_mẩn


## 4. Kiểm tra một câu cụ thể

Có thể thay đổi biến `sample_text` để kiểm tra pipeline.

In [3]:
sample_text = "Đồ ăn không ngon, giá hợp lý nhưng phục vụ hơi chậm."

result = pipeline.process(sample_text)

for key in ["raw", "cleaned", "normalized", "phobert_text"]:
    print(f"\n[{key.upper()}]")
    print(result.get(key))


[RAW]
Đồ ăn không ngon, giá hợp lý nhưng phục vụ hơi chậm.

[CLEANED]
Đồ ăn không ngon, giá hợp lý nhưng phục vụ hơi chậm.

[NORMALIZED]
Đồ ăn không_ngon , giá hợp lý nhưng phục vụ hơi chậm .

[PHOBERT_TEXT]
Đồ ăn_không_ngon , giá hợp_lý nhưng phục_vụ hơi chậm .


## 5. Chạy preprocessing cho toàn bộ dataset

Mặc định cell này **không chạy lại toàn bộ dataset** để tránh mất thời gian khi mở notebook.  
Khi cần tái tạo dữ liệu processed, đổi `RUN_FULL_PREPROCESS = True`.

In [4]:
RUN_FULL_PREPROCESS = False

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

if RUN_FULL_PREPROCESS:
    exported = pipeline.process_dataset(
        raw_dir=str(RAW_DIR),
        processed_dir=str(PROCESSED_DIR),
        keep_intermediate=False,
        output_column="phobert_text",
        aspect_output_column="aspect_sentiment",
    )
    print("Đã xuất:", exported)
else:
    print("Đang để RUN_FULL_PREPROCESS = False.")
    print("Khi cần chạy thật, đổi thành True rồi chạy lại cell.")

Đang để RUN_FULL_PREPROCESS = False.
Khi cần chạy thật, đổi thành True rồi chạy lại cell.


## 6. Đọc dữ liệu sau tiền xử lý

Nếu đã chạy `src/train.py --reprocess` hoặc cell trên, file processed là:

```text
data/processed/VLSP2018-ABSA-Restaurant.parquet
```

In [5]:
def read_processed_data() -> pd.DataFrame:
    files = sorted(PROCESSED_DIR.glob("*.parquet"))
    if not files:
        print("Chưa có file parquet trong data/processed.")
        return pd.DataFrame()
    # Ưu tiên file VLSP nếu có
    vlsp_files = [p for p in files if "VLSP2018" in p.name]
    path = vlsp_files[0] if vlsp_files else files[0]
    print("Đọc file:", path)
    return pd.read_parquet(path)

processed_df = read_processed_data()
print("Shape:", processed_df.shape)
display(processed_df.head(3))

Đọc file: C:\Users\ADMIN\Downloads\Food_Review_NLP\data\processed\VLSP2018-ABSA-Restaurant.parquet
Shape: (4751, 17)


,Review,AMBIENCE#GENERAL,DRINKS#PRICES,DRINKS#QUALITY,DRINKS#STYLE&OPTIONS,FOOD#PRICES,FOOD#QUALITY,FOOD#STYLE&OPTIONS,LOCATION#GENERAL,RESTAURANT#GENERAL,RESTAURANT#MISCELLANEOUS,RESTAURANT#PRICES,SERVICE#GENERAL,type,dataset,phobert_text,aspect_sentiment
0,"_ Ảnh chụp từ hôm qua, đi chơi với gia đình và 1 nhà họ hàng đang sống tại Sài Gòn. _ Hôm qua đi ăn trưa muộn, ai cũng đói hết nên lúc có đồ ăn là nhào vô ă...",0,0,0,0,0,3,3,0,0,0,0,0,train,VLSP2018-ABSA-Restaurant,"_ Ảnh chụp từ hôm_qua , đi chơi với gia_đình và 1 nhà họ_hàng đang sống tại Sài_Gòn . _ Hôm_qua đi ăn trưa muộn , ai cũng đói hết nên lúc có đồ_ăn là nhào v...","{""FOOD#QUALITY"": 3, ""FOOD#STYLE&OPTIONS"": 3}"
1,"_Hương vị thơm ngon, ăn cay cay rất thích, nêm nếm vừa miệng. Ngoài ra menu quán cũng nhiều món khác nhau tha hồ cho bạn lựa chọn luôn._Quán rộng rãi, view ...",1,0,0,0,3,1,1,0,1,0,3,2,train,VLSP2018-ABSA-Restaurant,"_Hương_vị thơm ngon , ăn cay_cay rất thích , nêm_nếm vừa miệng . Ngoài_ra thực_đơn_quán cũng nhiều món khác nhau tha_hồ cho bạn lựa_chọn luôn . _Quán rộng_r...","{""AMBIENCE#GENERAL"": 1, ""FOOD#PRICES"": 3, ""FOOD#QUALITY"": 1, ""FOOD#STYLE&OPTIONS"": 1, ""RESTAURANT#GENERAL"": 1, ""RESTAURANT#PRICES"": 3, ""SERVICE#GENERAL"": 2}"
2,"- 1 bàn tiệc hoành tráng 3 đứa ăn no muốn tắt thở mà giá chỉ 228k (ăn trung đợt giảm 10%), mình thích nhất pad thái vs gà nướng - đúng kiểu Thái luôn, quán ...",2,0,0,0,1,1,1,2,1,1,0,1,train,VLSP2018-ABSA-Restaurant,"- 1 bàn tiệc hoành_tráng 3 đứa ăn_no muốn tắt_thở mà giá chỉ 228 k ( ăn trung_đợt giảm 10 say_xỉn , mình thích nhất pad_thái với gà_nướng - đúng kiểu Thái l...","{""AMBIENCE#GENERAL"": 2, ""FOOD#PRICES"": 1, ""FOOD#QUALITY"": 1, ""FOOD#STYLE&OPTIONS"": 1, ""LOCATION#GENERAL"": 2, ""RESTAURANT#GENERAL"": 1, ""RESTAURANT#MISCELLANE..."


## 7. Kiểm tra chất lượng dữ liệu processed

Các kiểm tra chính:
- `phobert_text` không rỗng
- Các cột aspect có giá trị hợp lệ `0/1/2/3`
- split `train/dev(validation)/test` vẫn được giữ nguyên

In [6]:
def get_aspect_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if "#" in str(c)]

if not processed_df.empty:
    aspect_cols = get_aspect_columns(processed_df)

    checks = {
        "Số dòng": len(processed_df),
        "Có cột phobert_text": "phobert_text" in processed_df.columns,
        "phobert_text rỗng": int((processed_df["phobert_text"].fillna("").astype(str).str.strip() == "").sum())
            if "phobert_text" in processed_df.columns else None,
        "Số aspect": len(aspect_cols),
        "Có cột type": "type" in processed_df.columns,
    }
    display(pd.DataFrame(checks.items(), columns=["Kiểm tra", "Kết quả"]))

    invalid_rows = []
    for col in aspect_cols:
        values = set(processed_df[col].dropna().astype(int).unique())
        invalid = sorted(values - {0, 1, 2, 3})
        if invalid:
            invalid_rows.append({"aspect": col, "invalid_values": invalid})
    display(pd.DataFrame(invalid_rows) if invalid_rows else pd.DataFrame([{"Kết quả": "Tất cả nhãn aspect hợp lệ 0/1/2/3"}]))

    if "type" in processed_df.columns:
        display(processed_df["type"].value_counts().rename_axis("split").reset_index(name="count"))

,Kiểm tra,Kết quả
0,Số dòng,4751
1,Có cột phobert_text,True
2,phobert_text rỗng,0
3,Số aspect,12
4,Có cột type,True


,Kết quả
0,Tất cả nhãn aspect hợp lệ 0/1/2/3


,split,count
0,train,2961
1,dev,1290
2,test,500


## 8. So sánh trước và sau tiền xử lý

So sánh trực quan trước/sau tiền xử lý

In [7]:
if not processed_df.empty and "Review" in processed_df.columns and "phobert_text" in processed_df.columns:
    sample_show = processed_df[["Review", "phobert_text"]].sample(
        n=min(5, len(processed_df)),
        random_state=42,
    )
    display(sample_show)
else:
    print("Không đủ cột Review/phobert_text để hiển thị.")

,Review,phobert_text
3817,"Bát phở tình thương mến thương. Miếng gầu bò to như cái mặt mình vậy 😳 Bên cơ sở này chủ và nhân viên đều dễ thương, quán cũng sạch sẽ thoáng đãng hơn bên T...","Bát phở tình_thương mến_thương . Miếng gầu bò to như cái mặt mình vậy bất_ngờ Bên cơ_sở này chủ và nhân_viên đều dễ_thương , quán cũng sạch_sẽ thoáng đãng h..."
1075,"Gần 12h đêm thèm bia thì tìm được quán này, chỉ là rẽ đại thôi mà ưng ý hết sức. Do đã no nên 2 đứa chọn cái lẩu nấm hải sản - có vẻ nhẹ nhàng cơ mà bưng ra...","Gần 12 h đêm thèm bia thì tìm được quán này , chỉ là rẽ đại_thôi mà ưng_ý hết_sức . Do đã no nên 2 đứa chọn cái lẩu_nấm hải_sản - có_vẻ nhẹ_nhàng cơ mà bưng..."
296,Ăn no cành hong luôn ❤️❤️❤️❤️Ngon cực kỳ,Ăn no cành không_luôn yêu_thích ️_yêu_thích ️_yêu_thích ️_yêu_thích ️_Ngon cực_kỳ
2045,"Nhà hàng mới khai trương thôi nè, đang có buffet hơn 70 món lận đó, lại còn giảm 20% nha còn 239k thôi. Ngoài thịt bò thì còn có cả hải sản rồi thịt các loạ...","Nhà_hàng mới khai_trương thôi nè , đang có tiệc_đứng hơn 70 món lận đó , lại còn giảm 20 % nha còn 239 k thôi . Ngoài thịt bò thì còn có cả hải_sản rồi thịt..."
1703,"Món ngon nức tiếng Hà Thành. Ăn ở số 14 phố Chả Cá ngon hơn ở 107 Nguyễn Trường Tộ dù cùng một hệ thống. Món ăn thanh cảnh, ăn chơi sang miệng của người Hà ...","Món ngon_nức tiếng Hà_Thành . Ăn ở số 14 phố không_Cá ngon hơn ở 107 Nguyễn_Trường_Tộ dù cùng một hệ_thống . món_ăn thanh_cảnh , ăn_chơi sang miệng của ngườ..."


## 9. Nhận xét

Pipeline tiền xử lý giúp:
- Đồng nhất Unicode và khoảng trắng
- Chuẩn hóa một phần ngôn ngữ mạng xã hội
- Tạo input phù hợp với PhoBERT bằng word segmentation
- Giữ lại tín hiệu cảm xúc như phủ định, emoji, dấu câu
- Không làm mất cấu trúc ABSA của dataset

Điểm quan trọng: preprocessing không được biến bài toán ABSA thành sentiment classification đơn nhãn 
Ta cần giữ 12 cột aspect để model học từng khía cạnh riêng biệt